In [13]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)

# Define realistic Ayurvedic mappings
therapy_map = {
    "Anxiety": ["Shirodhara", "Abhyanga", "Meditation"],
    "Insomnia": ["Shirodhara", "Nasya", "Yoga Therapy"],
    "Joint Pain": ["Panchakarma", "Kizhi", "Abhyanga"],
    "Digestive Issues": ["Virechana", "Herbal Therapy", "Diet Correction"],
    "Stress": ["Shirodhara", "Meditation", "Abhyanga"],
    "Skin Disorders": ["Vamana", "Herbal Therapy", "Detox Therapy"],
    "Obesity": ["Udwarthanam", "Virechana", "Diet Correction"],
    "Diabetes": ["Vamana", "Herbal Therapy", "Yoga Therapy"],
    "Migraine": ["Nasya", "Shirodhara", "Abhyanga"],
    "Back Pain": ["Panchakarma", "Kizhi", "Abhyanga"]
}

dosha_map = {
    "Anxiety": "Vata",
    "Insomnia": "Vata",
    "Joint Pain": "Vata",
    "Digestive Issues": "Pitta",
    "Stress": "Pitta",
    "Skin Disorders": "Pitta",
    "Obesity": "Kapha",
    "Diabetes": "Kapha",
    "Migraine": "Vata",
    "Back Pain": "Vata"
}

genders = ["Male", "Female"]
lifestyles = ["Sedentary", "Moderate", "Active"]
stress_levels = ["Low", "Medium", "High"]

data = []

for i in range(500):
    age = np.random.randint(18, 70)
    gender = np.random.choice(genders)
    disease = np.random.choice(list(therapy_map.keys()))
    dosha = dosha_map[disease]
    severity = np.random.randint(1, 11)
    lifestyle = np.random.choice(lifestyles)
    stress = np.random.choice(stress_levels, p=[0.3, 0.5, 0.2])
    
    # Logical therapy assignment
    if np.random.rand() < 0.85:
        therapy = np.random.choice(therapy_map[disease])
    else:
        therapy = np.random.choice(sum(therapy_map.values(), []))
    
    data.append([age, gender, disease, dosha, lifestyle, stress, severity, therapy])

df = pd.DataFrame(data, columns=[
    "Age", "Gender", "Disease", "Dominant_Dosha", "Lifestyle", "Stress_Level", "Severity_Score", "Recommended_Therapy"
])
df.to_csv("ayurvedic_patients.csv", index=False)
print(" 'ayurvedic_patients.csv' created with 500 realistic Ayurveda records.")



# Load dataset
df = pd.read_csv("ayurvedic_patients.csv")

# Encode categorical features
encoders = {}
for col in ["Gender", "Disease", "Dominant_Dosha", "Lifestyle", "Stress_Level", "Recommended_Therapy"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Select meaningful features
features = ["Disease", "Dominant_Dosha", "Severity_Score", "Age"]
X = df[features]
y = df["Recommended_Therapy"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# GridSearch for optimal RandomForest parameters
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [6, 8, 10, None],
    'min_samples_split': [2, 4],
}
model = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
acc = round(accuracy_score(y_test, y_pred)*100, 2)
print(f" Best Parameters: {model.best_params_}")
print(f" Model Accuracy: {acc}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=1))

# Save model and encoders
joblib.dump(model.best_estimator_, "ayurveda_model.pkl")
for k, v in encoders.items():
    joblib.dump(v, f"encoder_{k}.pkl")

print("Model and encoders saved successfully!")

def predict_ayurvedic_therapy(age, gender, disease, dosha, lifestyle, stress, severity):
    model = joblib.load("ayurveda_model.pkl")
    enc_disease = joblib.load("encoder_Disease.pkl")
    enc_dosha = joblib.load("encoder_Dominant_Dosha.pkl")
    enc_therapy = joblib.load("encoder_Recommended_Therapy.pkl")

    # Handle unseen values
    if disease not in enc_disease.classes_:
        disease = enc_disease.classes_[0]
    if dosha not in enc_dosha.classes_:
        dosha = enc_dosha.classes_[0]

    disease_enc = enc_disease.transform([disease])[0]
    dosha_enc = enc_dosha.transform([dosha])[0]

    new_data = pd.DataFrame([{
        "Disease": disease_enc,
        "Dominant_Dosha": dosha_enc,
        "Severity_Score": severity,
        "Age": age
    }])

    pred = model.predict(new_data)
    confidence = model.predict_proba(new_data).max() * 100
    therapy = enc_therapy.inverse_transform(pred)[0]
    return therapy, round(confidence, 2)


# User Input
print("🌿 Enter patient details for Ayurvedic Therapy Recommendation\n")
age = int(input("Age: "))
gender = input("Gender (Male/Female): ").capitalize()
disease = input("Disease (Anxiety/Insomnia/Diabetes/etc): ").title()
dosha = input("Dominant Dosha (Vata/Pitta/Kapha): ").capitalize()
lifestyle = input("Lifestyle (Sedentary/Moderate/Active): ").capitalize()
stress = input("Stress Level (Low/Medium/High): ").capitalize()
severity = int(input("Severity Score (1–10): "))

therapy, conf = predict_ayurvedic_therapy(age, gender, disease, dosha, lifestyle, stress, severity)

print("\n💡 Recommended Ayurvedic Therapy:")
print("===================================")
print(f"👤 Age: {age}")
print(f"🚻 Gender: {gender}")
print(f"🩺 Disease: {disease}")
print(f"🧘 Dominant Dosha: {dosha}")
print(f"🔥 Severity: {severity}")
print("-----------------------------------")
print(f"🌿 Suggested Therapy: {therapy}")
print(f"📊 Confidence: {conf}%")
print("===================================")


 'ayurvedic_patients.csv' created with 500 realistic Ayurveda records.
 Best Parameters: {'max_depth': 6, 'min_samples_split': 2, 'n_estimators': 100}
 Model Accuracy: 34.0%

Classification Report:
               precision    recall  f1-score   support

           0       0.31      0.50      0.38        18
           1       1.00      0.00      0.00         2
           2       1.00      0.00      0.00         6
           3       0.36      0.50      0.42        10
           4       1.00      0.14      0.25         7
           5       0.00      0.00      0.00         8
           6       1.00      0.00      0.00         5
           7       0.40      0.50      0.44         8
           8       0.28      0.36      0.31        14
           9       1.00      0.00      0.00         2
          10       0.25      0.17      0.20         6
          11       0.47      1.00      0.64         8
          12       0.33      0.17      0.22         6

    accuracy                           0.34

Age:  24
Gender (Male/Female):  male
Disease (Anxiety/Insomnia/Diabetes/etc):  anxiety
Dominant Dosha (Vata/Pitta/Kapha):  pitta
Lifestyle (Sedentary/Moderate/Active):  active
Stress Level (Low/Medium/High):  low
Severity Score (1–10):  5



💡 Recommended Ayurvedic Therapy:
👤 Age: 24
🚻 Gender: Male
🩺 Disease: Anxiety
🧘 Dominant Dosha: Pitta
🔥 Severity: 5
-----------------------------------
🌿 Suggested Therapy: Virechana
📊 Confidence: 29.07%


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

np.random.seed(42)

# Simplified & consistent Ayurvedic mapping
therapy_map = {
    "Anxiety": "Relaxation Therapy",
    "Insomnia": "Relaxation Therapy",
    "Stress": "Relaxation Therapy",
    "Joint Pain": "Massage Therapy",
    "Back Pain": "Massage Therapy",
    "Migraine": "Relaxation Therapy",
    "Skin Disorders": "Detox Therapy",
    "Digestive Issues": "Detox Therapy",
    "Obesity": "Detox Therapy",
    "Diabetes": "Holistic Therapy"
}

dosha_map = {
    "Anxiety": "Vata",
    "Insomnia": "Vata",
    "Stress": "Pitta",
    "Joint Pain": "Vata",
    "Back Pain": "Vata",
    "Migraine": "Vata",
    "Skin Disorders": "Pitta",
    "Digestive Issues": "Pitta",
    "Obesity": "Kapha",
    "Diabetes": "Kapha"
}

data = []
for _ in range(500):
    age = np.random.randint(18, 70)
    gender = np.random.choice(["Male", "Female"])
    disease = np.random.choice(list(therapy_map.keys()))
    dosha = dosha_map[disease]
    severity = np.random.randint(1, 11)
    lifestyle = np.random.choice(["Sedentary", "Moderate", "Active"])
    stress = np.random.choice(["Low", "Medium", "High"], p=[0.3, 0.5, 0.2])

    # Consistent therapy logic
    therapy = therapy_map[disease]

    data.append([age, gender, disease, dosha, lifestyle, stress, severity, therapy])

df = pd.DataFrame(data, columns=[
    "Age", "Gender", "Disease", "Dominant_Dosha", "Lifestyle", "Stress_Level", "Severity_Score", "Recommended_Therapy"
])

# Save dataset
df.to_csv("ayurvedic_patients_v2.csv", index=False)
print("✅ Dataset created: ayurvedic_patients_v2.csv")

# Encode categorical variables
le_therapy = LabelEncoder()
df["Recommended_Therapy"] = le_therapy.fit_transform(df["Recommended_Therapy"])
joblib.dump(le_therapy, "encoder_Recommended_Therapy.pkl")

X = pd.get_dummies(df[["Disease", "Dominant_Dosha", "Severity_Score", "Age"]])
y = df["Recommended_Therapy"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train model
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)*100
print(f"🎯 Accuracy: {acc:.2f}%")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))

joblib.dump(model, "ayurveda_model_v2.pkl")
print("💾 Model saved successfully!")

def predict_ayurvedic_therapy(age, gender, disease, dosha, lifestyle, stress, severity):
    model = joblib.load("ayurveda_model.pkl")
    enc_disease = joblib.load("encoder_Disease.pkl")
    enc_dosha = joblib.load("encoder_Dominant_Dosha.pkl")
    enc_therapy = joblib.load("encoder_Recommended_Therapy.pkl")

    # Handle unseen values
    if disease not in enc_disease.classes_:
        disease = enc_disease.classes_[0]
    if dosha not in enc_dosha.classes_:
        dosha = enc_dosha.classes_[0]

    disease_enc = enc_disease.transform([disease])[0]
    dosha_enc = enc_dosha.transform([dosha])[0]

    new_data = pd.DataFrame([{
        "Disease": disease_enc,
        "Dominant_Dosha": dosha_enc,
        "Severity_Score": severity,
        "Age": age
    }])

    pred = model.predict(new_data)
    confidence = model.predict_proba(new_data).max() * 100
    therapy = enc_therapy.inverse_transform(pred)[0]
    return therapy, round(confidence, 2)


# User Input
print("🌿 Enter patient details for Ayurvedic Therapy Recommendation\n")
age = int(input("Age: "))
gender = input("Gender (Male/Female): ").capitalize()
disease = input("Disease (Anxiety/Insomnia/Diabetes/etc): ").title()
dosha = input("Dominant Dosha (Vata/Pitta/Kapha): ").capitalize()
lifestyle = input("Lifestyle (Sedentary/Moderate/Active): ").capitalize()
stress = input("Stress Level (Low/Medium/High): ").capitalize()
severity = int(input("Severity Score (1–10): "))

therapy, conf = predict_ayurvedic_therapy(age, gender, disease, dosha, lifestyle, stress, severity)

print("\n💡 Recommended Ayurvedic Therapy:")
print("===================================")
print(f"👤 Age: {age}")
print(f"🚻 Gender: {gender}")
print(f"🩺 Disease: {disease}")
print(f"🧘 Dominant Dosha: {dosha}")
print(f"🔥 Severity: {severity}")
print("-----------------------------------")
print(f"🌿 Suggested Therapy: {therapy}")
print(f"📊 Confidence: {conf}%")
print("===================================")


✅ Dataset created: ayurvedic_patients_v2.csv
🎯 Accuracy: 100.00%

📊 Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        31
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        19
           3       1.00      1.00      1.00        41

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

💾 Model saved successfully!
🌿 Enter patient details for Ayurvedic Therapy Recommendation



Age:  23
Gender (Male/Female):  male
Disease (Anxiety/Insomnia/Diabetes/etc):  anxiety
Dominant Dosha (Vata/Pitta/Kapha):  vata
Lifestyle (Sedentary/Moderate/Active):  active
Stress Level (Low/Medium/High):  low
Severity Score (1–10):  5



💡 Recommended Ayurvedic Therapy:
👤 Age: 23
🚻 Gender: Male
🩺 Disease: Anxiety
🧘 Dominant Dosha: Vata
🔥 Severity: 5
-----------------------------------
🌿 Suggested Therapy: Relaxation Therapy
📊 Confidence: 34.82%
